In [2]:
# !pip install U langchain-huggingface lightfm langchain sentence-transformers faiss-cpu langchain_community datasets chromadb ragas

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
# from difflib import SequenceMatcher
from datasets import load_dataset
from tqdm import tqdm 
sys.path.append('../../')
from utils.memory_utils import reduce_mem_usage, process_in_chunks, save_in_chunks

user_data = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_CDs_and_Vinyl", split="full", trust_remote_code=True)
meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_CDs_and_Vinyl", split="full", trust_remote_code=True)

def flatten_lst(col):
    # flatten lists by joining their elements with a separator (e.g., ', ')
    col['features'] = ', '.join(col['features']) if isinstance(col['features'], list) else col['features']
    col['description'] = ', '.join(col['description']) if isinstance(col['description'], list) else col['description']
    col['categories'] = ', '.join(col['categories']) if isinstance(col['categories'], list) else col['categories']

    return col

flattened_meta = meta.map(flatten_lst)

user_df = process_in_chunks(user_data.to_pandas(), chunk_size=10000)
meta_df = process_in_chunks(flattened_meta.to_pandas(), chunk_size=10000)
# user_df = user_data.to_pandas().fillna(0)
# meta_df = flattened_meta.to_pandas().fillna(0)

Processed 0 rows. Current memory usage: 3669.80 MB
Processed 10000 rows. Current memory usage: 3674.52 MB
Processed 20000 rows. Current memory usage: 3679.16 MB
Processed 30000 rows. Current memory usage: 3683.95 MB
Processed 40000 rows. Current memory usage: 3688.42 MB
Processed 50000 rows. Current memory usage: 3693.31 MB
Processed 60000 rows. Current memory usage: 3662.27 MB
Processed 70000 rows. Current memory usage: 3602.17 MB
Processed 80000 rows. Current memory usage: 3571.69 MB
Processed 90000 rows. Current memory usage: 3578.45 MB
Processed 100000 rows. Current memory usage: 3584.86 MB
Processed 110000 rows. Current memory usage: 3591.45 MB
Processed 120000 rows. Current memory usage: 3588.08 MB
Processed 130000 rows. Current memory usage: 3575.02 MB
Processed 140000 rows. Current memory usage: 3581.03 MB
Processed 150000 rows. Current memory usage: 3586.47 MB
Processed 160000 rows. Current memory usage: 3592.58 MB
Processed 170000 rows. Current memory usage: 3534.95 MB
Proces

In [4]:
user_df.head(5)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,1
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,1
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,1
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,1
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,0


In [5]:
user_df['verified_purchase'] = user_df['verified_purchase'].astype(int)
user_df.head(5)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,1
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,1
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,1
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,1
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,0


In [6]:
user_df['rating'] = user_df['rating'].astype('float32')
user_df.groupby(['rating','verified_purchase']).size().reset_index().pivot(columns='rating', index='verified_purchase', values=0)

rating,1.0,2.0,3.0,4.0,5.0
verified_purchase,,,,,
0,81604,63704,117798,288213,996369
1,106123,76853,165893,376301,2554415


In [7]:
user = user_df.drop(['images'], axis='columns')

In [8]:
user_df2 = user[['rating', 'asin', 'parent_asin', 'user_id', 'timestamp']]
user_df2

,rating,asin,parent_asin,user_id,timestamp
0,5.0,B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000
1,5.0,B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000
2,3.0,B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000
3,3.0,B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000
4,5.0,B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946
...,...,...,...,...,...
4827268,5.0,B000002VPH,B000002VPH,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308413175000
4827269,5.0,B000084T18,B000084T18,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308412875000
4827270,5.0,B004OFWLO0,B004OFWLO0,AHRJPHOI5JHYEQVSDMNX6736QH3Q,1505425559120
4827271,1.0,B000GIXIAK,B000GIXIAK,AH4PJ73QN75AJM5VSCT53AOADCGA,1157470110000


In [9]:
meta_df.head(5)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Digital Music,Release Some Tension,4.601562,112,,Swv ~ Release Some Tension,12.05,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SWV Format: Audio CD,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000002X4C,None,None,None
1,Digital Music,Rio Angie,5.000000,1,,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...",14.98,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Shrimp City Slim (Artist) Format: Audio CD,"CDs & Vinyl, Jazz, Avant Garde & Free Jazz","{""Product Dimensions"": ""5.6 x 0.4 x 4.9 inches...",B00902T10Y,None,None,None
2,Digital Music,Lost in Love,5.000000,9,,,24.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Nastyboy Klick Format: Audio CD,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore","{""Package Dimensions"": ""4.7 x 4.6 x 0.1 inches...",B00000DALY,None,None,None
3,Digital Music,Somewhere in Time,4.800781,1186,,The 1980 soundtrack to the now classic motion ...,11.55,"{'hi_res': [None, None], 'large': ['https://m....","{'title': [], 'url': [], 'user_id': []}","John Barry (Composer), Barry, John (Comp...","CDs & Vinyl, Soundtracks, Movie Scores","{""Is Discontinued By Manufacturer"": ""No"", ""Lan...",B0000086D1,None,None,None
4,Digital Music,Kimmon Waldruff,5.000000,1,,Solo acoustic fingerstyle guitar.,14.07,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Kimmon Waldruff (Artist) Format: Audio CD,"CDs & Vinyl, Folk","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000S6W7KC,None,None,None


In [10]:
meta_df['main_category'].unique()

array(['Digital Music', 'Movies & TV', 'Cell Phones & Accessories',
       'Books', 'Musical Instruments', 'Tools & Home Improvement',
       'Amazon Home', 'All Electronics', 'Toys & Games',
       'Health & Personal Care', 'Office Products', None, 'Software',
       'Home Audio & Theater', 'Video Games', 'Audible Audiobooks',
       'Industrial & Scientific', 'Grocery', 'All Beauty',
       'Arts, Crafts & Sewing', 'Sports & Outdoors', 'Pet Supplies',
       'Baby', 'Camera & Photo', 'Appliances', 'Computers',
       'AMAZON FASHION', 'Collectible Coins', 'Entertainment',
       'Automotive'], dtype=object)

In [11]:
meta_df.shape

(701959, 16)

In [12]:
meta_df = meta_df.drop_duplicates(subset=['parent_asin'],keep='last')
meta_df.shape

(701959, 16)

In [13]:
# drop cols dont provide contextual meaning
meta = meta_df.drop(['images', 'features', 'videos', 'store', 'price', 'details', 'bought_together', 'subtitle', 'author'], axis='columns')
meta

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,main_category,title,average_rating,rating_number,description,categories,parent_asin
0,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",B000002X4C
1,Digital Music,Rio Angie,5.000000,1,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...","CDs & Vinyl, Jazz, Avant Garde & Free Jazz",B00902T10Y
2,Digital Music,Lost in Love,5.000000,9,,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore",B00000DALY
3,Digital Music,Somewhere in Time,4.800781,1186,The 1980 soundtrack to the now classic motion ...,"CDs & Vinyl, Soundtracks, Movie Scores",B0000086D1
4,Digital Music,Kimmon Waldruff,5.000000,1,Solo acoustic fingerstyle guitar.,"CDs & Vinyl, Folk",B000S6W7KC
...,...,...,...,...,...,...,...
701954,Digital Music,Forever Gold: British Invasion,3.000000,1,British Invasion ~ Forever Gold: British Invasion,"CDs & Vinyl, International Music, Europe, Brit...",B00005AVHN
701955,Digital Music,"Joan Hammond, Historical Recordings from 1941-49",5.000000,1,,"CDs & Vinyl, Classical",B000T001IM
701956,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout...",B00069I6RO
701957,Digital Music,Long Day in the Milky Way,4.398438,16,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk",B08BF2PH1X


In [14]:
merge = user_df2.merge(meta, on='parent_asin', how='right')
merge

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,asin,parent_asin,user_id,timestamp,main_category,title,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
3,5.0,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,1.427480e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
4,2.0,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,1.289638e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
...,...,...,...,...,...,...,...,...,...,...,...
4827554,5.0,B000T001IM,B000T001IM,AFICGLJ62Y3YRTYN7PNYM6UUOA6Q,1.387540e+12,Digital Music,"Joan Hammond, Historical Recordings from 1941-49",5.000000,1,,"CDs & Vinyl, Classical"
4827555,4.0,B00069I6RO,B00069I6RO,AHLRFMDIVAQF7ARTPM6PVJO6DTYQ,1.199348e+12,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout..."
4827556,5.0,B00069I6RO,B00069I6RO,AGZVQUGBE64F3M2A5EBU2CNDTKIA,1.103150e+12,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout..."
4827557,5.0,B08BF2PH1X,B08BF2PH1X,AEZET6CFIEJWTU7UTEK57IDARX2A,1.612912e+12,Digital Music,Long Day in the Milky Way,4.398438,16,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk"


In [15]:
date_min = pd.to_datetime(merge.timestamp,unit='ms').min()
date_max = pd.to_datetime(merge.timestamp,unit='ms').max()
date_min, date_max

(Timestamp('1997-09-09 03:13:17'), Timestamp('2023-09-09 15:33:39.315000'))

In [16]:
date_gap = (date_max-date_min)//(26*2)
date_gap

Timedelta('182 days 15:00:23.506057692')

In [17]:
merge_df2 = merge.copy()
merge_df2.columns

Index(['rating', 'asin', 'parent_asin', 'user_id', 'timestamp',
       'main_category', 'title', 'average_rating', 'rating_number',
       'description', 'categories'],
      dtype='object')

In [18]:
merge_df2.columns = ['rating', 'iid', 'parent_iid', 'uid', 'timestamp',
       'main_category', 'title', 'average_rating', 'rating_number',
       'description', 'categories']
merge_df2.head(2)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"


In [19]:
merge_df2.shape

(4827559, 11)

In [20]:
merge_df2['year'] = pd.to_datetime(merge_df2['timestamp'], unit='ms').dt.year
merge_df2.head(2)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories,year
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2015.0
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2013.0


In [21]:
merge_df2 = merge_df2.dropna()
merge_df2.shape

(4823633, 12)

In [22]:
merge_df2.iid.nunique(), merge_df2.parent_iid.unique(), merge_df2.uid.nunique()

(700877,
 array(['B000002X4C', 'B00902T10Y', 'B00000DALY', ..., 'B00069I6RO',
        'B08BF2PH1X', 'B000001LW6'], shape=(700844,), dtype=object),
 1752912)

In [23]:
np.sort(merge_df2.year.unique())

array([1997., 1998., 1999., 2000., 2001., 2002., 2003., 2004., 2005.,
       2006., 2007., 2008., 2009., 2010., 2011., 2012., 2013., 2014.,
       2015., 2016., 2017., 2018., 2019., 2020., 2021., 2022., 2023.])

In [24]:
#  number of ratings given by each user in the s_rating
merge_df2.groupby('uid').agg({"rating":'count'})

,rating
uid,
AE22232OB6S7UAC75JUFYRGBSHIQ,1
AE22236AFRRSMQIKGG7TPTB75QEA,4
AE222GY2VH7MEWI4N5AHPXB5CP5A,1
AE222H3FGXWLHRFUMGMS2RR57NDQ,8
AE222K4ISB45WWMKN7VSYH5WBNNQ,2
...,...
AHZZZSX4HTKLTFHCS4MEXKCNMJAA,1
AHZZZWVHLYTQU55PD4FJULUEKTXA,1
AHZZZZ3CH2ZFY4CLZTVGAJYCAWIA,1


In [25]:
item_info = merge_df2.groupby('iid').agg({"rating_number": 'first'})
user_info = merge_df2.groupby('uid').agg({"rating":'count'})

In [26]:
# filters the user_info, item_info DF to select items that have received more than 20 ratings
active_item = item_info[item_info['rating_number']>20].index
active_user = user_info[user_info['rating']>20].index
active_item.shape, active_user.shape

((258651,), (21839,))

In [39]:
# filters the item_info df to select items that have received more than 20 ratings
# .index is then used to retrieve the IDs of these active items
merge_df3 = merge_df2[merge_df2['uid'].isin(active_user)]
print(merge_df3.shape)
merge_df3 = merge_df2[merge_df2['iid'].isin(active_item)]
print(merge_df3.shape)

(1165602, 12)
(3937092, 12)


In [40]:
item_info2 = merge_df3.groupby('iid').agg({"rating_number": 'first'})
user_info2 = merge_df3.groupby('uid').agg({"rating":'count'})
user_info2.mean(), item_info2.mean()

(rating    2.526467
 dtype: float64,
 rating_number    289.008289
 dtype: float64)

In [41]:
merge_df3 = merge_df2[merge_df2['iid'].isin(active_item)]
merge_df3.shape

(3937092, 12)

In [42]:
merge_df3.iid.nunique(), merge_df3.parent_iid.nunique(), merge_df3.uid.nunique()

(258651, 258637, 1558339)

In [37]:
merge_df3.head(3)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories,year
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2015.0
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2013.0
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2000.0


In [43]:
merge_df3.groupby('year').agg({'rating':'count'}).reset_index().plot(x='year',kind='bar')

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

In [44]:
merge_df3['label'] = 0
merge_df3.loc[merge_df3['rating'] >= 4, 'label'] = 1
merge_df3['label'].describe()

/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_99256/2042806392.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df3['label'] = 0


count    3.937092e+06
mean     8.744667e-01
std      3.313226e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: label, dtype: float64

In [32]:
# creates mappings for user IDs and item IDs to consecutive numerical values, starting from 1
users = merge_df3.uid.unique()
items = merge_df3.iid.unique()

# creates dicts users_map, items_map where each unique uid, iid is mapped to a consecutive numerical value starting from 1
users_map = dict(zip(users, np.arange(users.shape[0])+1))
items_map = dict(zip(items, np.arange(items.shape[0])+1))

merge_df3['uid'] = merge_df3['uid'].map(users_map)
merge_df3['iid'] = merge_df3['iid'].map(items_map)
merge_df3.uid.max(), merge_df3.iid.max()

/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_94953/1897254940.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df3['uid'] = merge_df3['uid'].map(users_map)
/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_94953/1897254940.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df3['iid'] = merge_df3['iid'].map(items_map)


(1558339, 258651)

In [33]:
merge_df3.head(3)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories,year,label
0,1.0,1,B000002X4C,1,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2015.0,0
1,1.0,1,B000002X4C,2,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2013.0,0
2,4.0,1,B000002X4C,3,9.534267e+11,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2000.0,1


In [34]:
merge_df3['time'] = ((pd.to_datetime(merge_df3['timestamp'], unit='ms') - date_min) // date_gap).astype(int)
merge_df3.head(5)

/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_94953/3529764540.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df3['time'] = ((pd.to_datetime(merge_df3['timestamp'], unit='ms') - date_min) // date_gap).astype(int)
/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories,year,label,time
0,1.0,1,B000002X4C,1,1.420266e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2015.0,0,34
1,1.0,1,B000002X4C,2,1.379363e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2013.0,0,32
2,4.0,1,B000002X4C,3,9.534267e+11,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2000.0,1,5
3,5.0,1,B000002X4C,4,1.427480e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2015.0,1,35
4,2.0,1,B000002X4C,5,1.289638e+12,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",2010.0,0,26


In [35]:
#  splits the merge_df3 into train, val and test sets based on time
rating_train = merge_df3[merge_df3['time'].isin(range(2 * 26 - 2))].copy()
rating_valid_test = merge_df3[merge_df3['time'] == 2 * 26 - 2].copy()
rating_valid_test.sort_values(by='timestamp', inplace=True)
n = rating_valid_test.shape[0] // 2
rating_valid = rating_valid_test.iloc[:n].copy()
rating_test = rating_valid_test.iloc[n:].copy()
rating_train.shape, rating_valid.shape, rating_test.shape

((3862643, 14), (27538, 14), (27539, 14))

In [36]:
# identify cold-start, warm-start for val and test sets
# Adds a 'not_cold' column to rating_valid, marking interactions as 1 if both user and item exist in the training set, else 0
train_user = rating_train['uid'].unique()
train_item = rating_train['iid'].unique()
rating_valid['not_cold'] = rating_valid[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")
rating_test['not_cold'] = rating_test[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")

In [37]:
rating_train.shape, rating_valid.shape, rating_test.shape

((3862643, 14), (27538, 15), (27539, 15))

In [38]:
rating_valid.not_cold.value_counts(), rating_test.not_cold.value_counts()

(not_cold
 0    16396
 1    11142
 Name: count, dtype: int64,
 not_cold
 0    17649
 1     9890
 Name: count, dtype: int64)

In [39]:
# remove cold-start interactions from the validation and test sets
# cold-start interactions involve users or items that were not present in the training data
def filter_cold_start(train,valid,test):
    train_user = train.uid.unique()
    train_item = train.iid.unique()
    valid = valid[valid['uid'].isin(train_user)]
    test = test[test['uid'].isin(train_user)]
    valid = valid[valid['iid'].isin(train_item)]
    test = test[test['iid'].isin(train_item)]
    return valid, test

In [40]:
rating_train.label.mean(), rating_valid.label.mean(), rating_test.label.mean()

(np.float64(0.8750749163202501),
 np.float64(0.8467572082213668),
 np.float64(0.8432768074367261))

In [41]:
# processes interaction data for each user
# creating a list of tuples representing each interaction along with the user's history
import copy

def deal_with_each_u(x,u):
    items = np.array(x.iid)
    labels = np.array(x.label)
    titles = np.array(x.title)
    categories = list(x.categories)
    timestamp = np.array(x.timestamp)
    flags =  np.array(x.flag)# adding a '0' by default
    his = [0] # adding a '0' by default
    his_title = ['']
    his_cats = ['']
    results = []
    for i in range(items.shape[0]):
        results.append((u, items[i], timestamp[i], np.array(his), copy.copy(his_title), copy.copy(his_cats), titles[i], categories[i], labels[i], flags[i]))
        # training data
        #  If the interaction is positive, the item ID and title are added to the user's history lists
        if labels[i] > 0: # positive
            his.append(items[i])
            his_title.append(titles[i])
            his_cats.append(categories[i])
        # validation data
    return results

In [42]:
rating_valid_2 = rating_valid
rating_test_2 = rating_test

In [43]:
# adds a 'flag' column to the training, validation, and test sets
# assigns a value of -1 to all rows in this column, indicating that these rows belong to the training set
rating_train['flag'] =  pd.DataFrame(np.ones(rating_train.shape[0])*-1, index = rating_train.index) # 0 ~ valid set
rating_valid_2['flag'] = pd.DataFrame(np.zeros(rating_valid_2.shape[0]), index = rating_valid_2.index)  # 1 ~ test set
rating_test_2['flag'] = pd.DataFrame(np.ones(rating_test_2.shape[0]), index = rating_test_2.index)
data = pd.concat([rating_train, rating_valid_2, rating_test_2], axis = 0, ignore_index = True)
data = data.sort_values(by = ['uid','timestamp'])

# groups data df by user ID
# for each user group, it creates lists of item IDs ('iid'), labels, titles, description, timestamps, flags
u_inter_all = data.groupby('uid').agg({'iid':list, 'label':list, 'title':list, 'categories': list, 'timestamp':list,'flag':list})


In [44]:
data.flag.unique()

array([-1.,  1.,  0.])

In [45]:
data.head(5)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,main_category,title,average_rating,rating_number,description,categories,year,label,time,flag,not_cold
1745738,3.0,105479,B001QCKJAM,1,1.372449e+12,Digital Music,Xscape: Super Hits,4.699219,826,I will ship by EMS or SAL items in stock in Ja...,"CDs & Vinyl, Pop, Dance Pop",2013.0,0,31,-1.0,NaN
1724819,5.0,104236,B000063BQL,1,1.398165e+12,Digital Music,Ultimate Collection,4.800781,406,Johnny Gill Collection gathers his best solo h...,"CDs & Vinyl, R&B, Soul",2014.0,1,33,-1.0,NaN
2635765,5.0,158516,B000001E3R,1,1.400653e+12,Digital Music,Natural Thing,4.699219,40,,"CDs & Vinyl, R&B, Soul",2014.0,1,33,-1.0,NaN
562036,1.0,33730,B000251Y1C,1,1.413461e+12,Digital Music,Understanding,4.699219,50,CD,"CDs & Vinyl, Pop, Dance Pop",2014.0,0,34,-1.0,NaN
2696436,5.0,162105,B0002Y4SUC,1,1.414292e+12,Digital Music,Soulful Sounds of Christmas,4.601562,40,"Product Description, Get your holiday groove o...","CDs & Vinyl, Holiday & Wedding, Christmas",2014.0,1,34,-1.0,NaN


In [46]:
u_inter_all.head(5)

,iid,label,title,categories,timestamp,flag
uid,,,,,,
1,"[105479, 104236, 158516, 33730, 162105, 218784...","[0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[Xscape: Super Hits, Ultimate Collection, Natu...","[CDs & Vinyl, Pop, Dance Pop, CDs & Vinyl, R&B...","[1372448540000.0, 1398165102000.0, 14006530020...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
2,"[29331, 145502, 193578, 51302, 39011, 136252, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, ...","[Haunted, Mexican Moon, Ten, For Unlawful Carn...","[CDs & Vinyl, Pop, Singer-Songwriters, CDs & V...","[1282765017000.0, 1300378226000.0, 13014264660...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
3,"[183280, 188922, 215753, 42891, 84885, 151334,...","[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, ...","[1999, This Is Your Night, Design of a Decade ...","[CDs & Vinyl, Indie & Alternative, New Wave & ...","[946548860000.0, 946556194000.0, 947984281000....","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
4,"[215749, 109076, 45769, 154300, 201946, 163658...","[1, 1, 1, 1, 1, 1, 1, 1, 1]","[Piece of Mind Explicit Lyrics, Shabba R...","[CDs & Vinyl, R&B, Funk, CDs & Vinyl, Reggae, ...","[1356674276000.0, 1358400582000.0, 13584006580...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
5,"[214310, 53073, 81194, 197200, 124266, 33678, ...","[1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, ...","[Charmbracelet, The Soul Sessions, Javier, The...","[CDs & Vinyl, Pop, Adult Contemporary, CDs & V...","[1057855934000.0, 1066771811000.0, 10691320250...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."


In [47]:
# each user in the u_inter_all df  
# apply the deal_with_each_u function to process their interaction data
# retrieves the row corresponding to the current uid and uid then add it to results
results = []
for u in u_inter_all.index:
    results.extend(deal_with_each_u(u_inter_all.loc[u], u))

In [48]:
len(results)

3917720

In [49]:
u, i, time, label, his, his_title, his_cats, title, categories, flag = [], [], [], [], [], [], [], [], [], []
for re in results:
    u.append(re[0])
    i.append(re[1])
    time.append(re[2])
    his.append(re[3])
    his_title.append(re[4])
    his_cats.append(re[5])
    title.append(re[6])
    categories.append(re[7])
    label.append(re[8])
    flag.append(re[9])

In [50]:
data = pd.DataFrame({"uid": u,'iid': i,'label': label, 'timestamp': time, 'his': his,'his_title': his_title, 'his_cats': his_cats, 'title': title, 'categories': categories, 'flag': flag})

In [51]:
data.head(5)

,uid,iid,label,timestamp,his,his_title,his_cats,title,categories,flag
0,1,105479,0,1.372449e+12,[0],[],[],Xscape: Super Hits,"CDs & Vinyl, Pop, Dance Pop",-1.0
1,1,104236,1,1.398165e+12,[0],[],[],Ultimate Collection,"CDs & Vinyl, R&B, Soul",-1.0
2,1,158516,1,1.400653e+12,"[0, 104236]","[, Ultimate Collection]","[, CDs & Vinyl, R&B, Soul]",Natural Thing,"CDs & Vinyl, R&B, Soul",-1.0
3,1,33730,0,1.413461e+12,"[0, 104236, 158516]","[, Ultimate Collection, Natural Thing]","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Understanding,"CDs & Vinyl, Pop, Dance Pop",-1.0
4,1,162105,1,1.414292e+12,"[0, 104236, 158516]","[, Ultimate Collection, Natural Thing]","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Soulful Sounds of Christmas,"CDs & Vinyl, Holiday & Wedding, Christmas",-1.0


In [52]:
data.tail(5)

,uid,iid,label,timestamp,his,his_title,his_cats,title,categories,flag
3917715,1558335,258646,1,1.572986e+12,[0],[],[],My Love Is Your Love,"CDs & Vinyl, Pop, Adult Contemporary",-1.0
3917716,1558336,258648,1,1.537987e+12,[0],[],[],Streets of Gold explicit_lyrics,"CDs & Vinyl, Pop, Adult Alternative",-1.0
3917717,1558337,258649,1,1.525041e+12,[0],[],[],EXO-K Mama 1st Mini Album CD Korean Language K...,"CDs & Vinyl, Pop, Korean Pop",-1.0
3917718,1558338,258649,1,1.447653e+12,[0],[],[],EXO-K Mama 1st Mini Album CD Korean Language K...,"CDs & Vinyl, Pop, Korean Pop",-1.0
3917719,1558339,258649,1,1.515795e+12,[0],[],[],EXO-K Mama 1st Mini Album CD Korean Language K...,"CDs & Vinyl, Pop, Korean Pop",-1.0


In [53]:
data.label.describe()

count    3.917720e+06
mean     8.746523e-01
std      3.311127e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: label, dtype: float64

In [54]:
train = data[data['flag'].isin([-1])].copy()
val = data[data['flag'].isin([0])].copy()
test = data[data['flag'].isin([1])].copy()
train.shape,val.shape,test.shape

((3862643, 10), (27538, 10), (27539, 10))

In [55]:
train_user = train['uid'].unique()
train_item = train['iid'].unique()
val['not_cold'] = val[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")
test['not_cold'] = test[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")

In [56]:
train['not_cold'] = pd.DataFrame(np.ones(train.shape[0]), index = train.index).astype("int")
train.head(2)

,uid,iid,label,timestamp,his,his_title,his_cats,title,categories,flag,not_cold
0,1,105479,0,1.372449e+12,[0],[],[],Xscape: Super Hits,"CDs & Vinyl, Pop, Dance Pop",-1.0,1
1,1,104236,1,1.398165e+12,[0],[],[],Ultimate Collection,"CDs & Vinyl, R&B, Soul",-1.0,1


In [57]:
train.shape, val.shape, test.shape

((3862643, 11), (27538, 11), (27539, 11))

In [58]:
train.head(10)

,uid,iid,label,timestamp,his,his_title,his_cats,title,categories,flag,not_cold
0,1,105479,0,1.372449e+12,[0],[],[],Xscape: Super Hits,"CDs & Vinyl, Pop, Dance Pop",-1.0,1
1,1,104236,1,1.398165e+12,[0],[],[],Ultimate Collection,"CDs & Vinyl, R&B, Soul",-1.0,1
2,1,158516,1,1.400653e+12,"[0, 104236]","[, Ultimate Collection]","[, CDs & Vinyl, R&B, Soul]",Natural Thing,"CDs & Vinyl, R&B, Soul",-1.0,1
3,1,33730,0,1.413461e+12,"[0, 104236, 158516]","[, Ultimate Collection, Natural Thing]","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Understanding,"CDs & Vinyl, Pop, Dance Pop",-1.0,1
4,1,162105,1,1.414292e+12,"[0, 104236, 158516]","[, Ultimate Collection, Natural Thing]","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Soulful Sounds of Christmas,"CDs & Vinyl, Holiday & Wedding, Christmas",-1.0,1
5,1,218784,1,1.419318e+12,"[0, 104236, 158516, 162105]","[, Ultimate Collection, Natural Thing, Soulful...","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Number Ones,"CDs & Vinyl, Pop, Adult Contemporary",-1.0,1
6,1,93473,1,1.419323e+12,"[0, 104236, 158516, 162105, 218784]","[, Ultimate Collection, Natural Thing, Soulful...","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Pure 80's Love: The #1 Hits,"CDs & Vinyl, Pop",-1.0,1
7,1,10494,1,1.419512e+12,"[0, 104236, 158516, 162105, 218784, 93473]","[, Ultimate Collection, Natural Thing, Soulful...","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",NKOTBSB,"CDs & Vinyl, Pop",-1.0,1
8,1,219864,1,1.419647e+12,"[0, 104236, 158516, 162105, 218784, 93473, 10494]","[, Ultimate Collection, Natural Thing, Soulful...","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Faith Evans,"CDs & Vinyl, Dance & Electronic, House",-1.0,1
9,1,146041,1,1.419647e+12,"[0, 104236, 158516, 162105, 218784, 93473, 104...","[, Ultimate Collection, Natural Thing, Soulful...","[, CDs & Vinyl, R&B, Soul, CDs & Vinyl, R&B, S...",Special Christmas,"CDs & Vinyl, Holiday & Wedding, Christmas",-1.0,1


In [59]:
shuffled_train = train.sample(frac=1, random_state=42).reset_index(drop=True)

# 10k and 100k interaction samples
sample_10k = shuffled_train.iloc[:10_000].copy()
sample_100k = shuffled_train.iloc[:100_000].copy()

In [61]:
save_to_path = "../../data/amazon/samples"
os.makedirs(save_to_path, exist_ok=True)

sample_10k.to_csv(os.path.join(save_to_path, "sample_10k.csv"), index=False)
sample_100k.to_csv(os.path.join(save_to_path, "sample_100k.csv"), index=False)
print("Saved 10k and 100k interaction samples.")

Saved 10k and 100k interaction samples.


In [58]:
# get sample of 100k users
unique_users = train['uid'].unique()
sampled_user_ids = pd.Series(unique_users).sample(n=100000, random_state=42)

sampled_train = train[train['uid'].isin(sampled_user_ids)].copy()

print(f"Sampled {len(sampled_user_ids)} users")
print(f"Resulting dataset has {len(sampled_train)} interactions")
print(f"Unique items in sampled set: {sampled_train['iid'].nunique()}")

Sampled 100000 users
Resulting dataset has 248383 interactions
Unique items in sampled set: 92781


In [59]:
# dataset = Dataset()
from lightfm import LightFM
from lightfm.data import Dataset


def fit_and_train(data, learning_rate, epochs):
    dataset = Dataset()
    dataset.fit(data['uid'], data['iid'])

    # build interaction matrices
    interactions, weights = dataset.build_interactions((row['uid'], row['iid'], row['label']) for id, row in data.iterrows())

    model = LightFM(loss='warp', learning_rate=learning_rate, item_alpha=1e-6, user_alpha=1e-6)
    # train model
    model.fit(interactions, epochs=epochs, num_threads=4, sample_weight=weights)
    return model, dataset, interactions
    
    # test_interactions, test_w = dataset.build_interactions((row['user_id_encoded'], row['asin_encoded']) for id, row in df_test.iterrows())

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [ ]:
def get_top_k(model, dataset, data, user_id, k):    
    n_users, n_items = dataset.interactions_shape()
    user_id_to_index, item_id_to_index, _, _ = dataset.mapping()
    user_index = user_id_to_index[user_id]
    rated_raw_items = set(data[data['uid'] == user_id]['iid'])

    # convert raw IDs to valid indices (skip unseen items)
    rated_indices = [
        item_id_to_index[i] 
        for i in rated_raw_items 
        if i in item_id_to_index 
    ]
    user_indices = np.full(n_items, user_index)
    item_indices = np.arange(n_items)
    scores = model.predict(user_indices, item_indices)
    
    # user_ids = np.full(n_items, user_id)  # or use np.repeat(user_id, n_items)
    # # item_ids = np.arange(n_items)
    # rated_items = set(data[data['uid']==user_id]['iid'])

    # scores = model.predict(user_ids, np.arange(n_items))
    # scores[list(rated_items)] = -np.inf
    # top_items = np.argsort(-scores)[:k]

    scores[rated_indices] = -np.inf

    # Get top-k items (as LightFM indices)
    top_indices = np.argsort(-scores)[:k]

    # Map top indices back to raw IDs for lookup
    index_to_item_id = {v: k for k, v in item_id_to_index.items()}
    top_raw_items = [index_to_item_id[i] for i in top_indices]

    items_lookup = {
        iid: {
            'iid': iid,
            'title': title,
            'categories': cat,
        }
        for iid, title, cat in zip(data['iid'], data['title'], data['categories'])
    }
    
    # recs = [
    #     {
    #         **items_lookup[i],
    #         'score': float(scores[i])
    #     }
    #     for i in top_items if i in items_lookup
    # ]

    recs = [{
            **items_lookup[iid],
            'score': float(scores[item_id_to_index[iid]])
        }
        for iid in top_raw_items if iid in items_lookup]
    
    # iid_info = dict(zip(data['iid'], zip(data['iid'], data['title'], data['description'])))
    # recs = [iid_info[item] for item in top_items if item in iid_info]
    return recs


# def get_recs(user_id):
#     return recs_dict.get(user_id, [])

In [60]:
def get_all_user_recommendations_batch(model, dataset, data, k=50, batch_size=100):
    """
    Optimized version using batch prediction for better performance
    """
    n_users, n_items = dataset.interactions_shape()
    user_id_to_index, item_id_to_index = dataset.mapping()[:2]
    
    # Get all unique user IDs from training data
    all_user_ids = data['uid'].unique()
    
    # Create reverse mapping
    index_to_item_id = {v: k for k, v in item_id_to_index.items()}
    
    user_recommendations = {}
    
    print(f"Processing {len(all_user_ids)} users in batches of {batch_size}")
    
    for i in tqdm(range(0, len(all_user_ids), batch_size)):
        batch_user_ids = all_user_ids[i:i+batch_size]
        
        # Filter valid users
        valid_users = [uid for uid in batch_user_ids if uid in user_id_to_index]
        
        if not valid_users:
            continue
            
        # Create user and item index arrays for batch prediction
        user_indices = np.array([user_id_to_index[uid] for uid in valid_users])
        
        # For each user, predict against all items
        for user_id in valid_users:
            try:
                user_index = user_id_to_index[user_id]
                
                # Get rated items
                rated_items = set(data[data['uid'] == user_id]['iid'])
                rated_indices = [item_id_to_index[i] for i in rated_items if i in item_id_to_index]
                
                # Predict for this user against all items
                user_array = np.full(n_items, user_index, dtype=np.int32)
                item_array = np.arange(n_items, dtype=np.int32)
                
                scores = model.predict(user_array, item_array)
                
                # Mask rated items
                scores[rated_indices] = -np.inf
                
                # Get top-k
                top_indices = np.argsort(-scores)[:k]
                
                # Convert to recommendations
                recommendations = []
                for idx in top_indices:
                    if idx in index_to_item_id:
                        raw_item_id = index_to_item_id[idx]
                        item_data = data[data['iid'] == raw_item_id]
                        if not item_data.empty:
                            item_row = item_data.iloc[0]
                            recommendations.append({
                                'iid': raw_item_id,
                                'title': item_row['title'],
                                'categories': item_row['categories'],
                                'score': float(scores[idx])
                            })
                
                user_recommendations[user_id] = recommendations
                
            except Exception as e:
                print(f"Error processing user {user_id}: {e}")
                continue
    
    return user_recommendations

In [61]:
 model, dataset, interactions = fit_and_train(sampled_train, learning_rate=0.01, epochs=50)

In [62]:
user_recs = get_all_user_recommendations_batch(model, dataset, sampled_train, k=50)
print(f"Generated recommendations for {len(user_recs)} users")
print(f"Sample user recommendations: {list(user_recs.keys())[:3]}")

Processing 100000 users in batches of 100


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [26:08<00:00,  1.57s/it]

Generated recommendations for 100000 users
Sample user recommendations: [np.int64(32), np.int64(34), np.int64(77)]


In [63]:
print(f"Sample user recommendations: {list(user_recs.values())[:5]}")

Sample user recommendations: [[{'iid': 170245, 'title': np.str_('Of Vengeance & Violence'), 'categories': 'CDs & Vinyl, Metal, Alternative Metal', 'score': 0.9593205451965332}, {'iid': 95724, 'title': np.str_('Colour the Small One'), 'categories': 'CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi, Indie Rock', 'score': 0.9240134954452515}, {'iid': 155333, 'title': np.str_('Pastel Blues'), 'categories': 'CDs & Vinyl, Broadway & Vocalists, Cabaret', 'score': 0.9088192582130432}, {'iid': 185782, 'title': np.str_('Wishbone Four'), 'categories': "CDs & Vinyl, Today's Deals in Music, Hard Rock & Metal, Hard Rock, CDs $7 - $10", 'score': 0.8685558438301086}, {'iid': 185493, 'title': np.str_('Roy Orbisons Greatest Hits'), 'categories': 'CDs & Vinyl, Pop', 'score': 0.8253015875816345}, {'iid': 1331, 'title': np.str_('Coral Fang       Explicit Lyrics'), 'categories': 'CDs & Vinyl, Indie & Alternative, Hardcore & Punk, Punk', 'score': 0.8169475197792053}, {'iid': 77088, 'title': np.str_('Hammered 

In [64]:
print("Top 50 recommendations for first 5 users:\n")

for i, (user_id, recs) in enumerate(user_recs.items()):
    if i >= 5:
        break
    print(f"user_id: {user_id}  top 50 is:")
    for rec in recs:
        print(f"  'iid': {rec['iid']}, 'title': {rec['title']}, 'categories': {rec['categories']}, 'score': {rec['score']}")
    print()

Top 50 recommendations for first 5 users:

user_id: 32  top 50 is:
  'iid': 170245, 'title': Of Vengeance & Violence, 'categories': CDs & Vinyl, Metal, Alternative Metal, 'score': 0.9593205451965332
  'iid': 95724, 'title': Colour the Small One, 'categories': CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi, Indie Rock, 'score': 0.9240134954452515
  'iid': 155333, 'title': Pastel Blues, 'categories': CDs & Vinyl, Broadway & Vocalists, Cabaret, 'score': 0.9088192582130432
  'iid': 185782, 'title': Wishbone Four, 'categories': CDs & Vinyl, Today's Deals in Music, Hard Rock & Metal, Hard Rock, CDs $7 - $10, 'score': 0.8685558438301086
  'iid': 185493, 'title': Roy Orbisons Greatest Hits, 'categories': CDs & Vinyl, Pop, 'score': 0.8253015875816345
  'iid': 1331, 'title': Coral Fang       Explicit Lyrics, 'categories': CDs & Vinyl, Indie & Alternative, Hardcore & Punk, Punk, 'score': 0.8169475197792053
  'iid': 77088, 'title': Hammered Dulcimer Christmas, 'categories': CDs & Vinyl, Holiday &

In [65]:
mappings = dataset.mapping()
print(f"Number of elements returned: {len(mappings)}")
print(f"Type of elements: {[type(m) for m in mappings]}")

Number of elements returned: 4
Type of elements: [<class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>]


In [66]:
print("Users:", len(dataset.mapping()[0]), "Items:", len(dataset.mapping()[1]))

Users: 100000 Items: 100000


In [66]:
recs = get_top_k(model, dataset, train, user_id=1518328, k=5)
for item in recs:
    print(f"ID: {item['iid']}, Title: {item['title']}, Categories: {item['categories']}, Score: {item['score']:.3f}")
    
# recommendations = get_top_k(model, dataset, interactions, train, user_id=1, k=10)
# print("Top recommendations")
# for item in recommendations:
#     print(f"{item['title']} (score: {item['score']:.3f})")

ID: 2150, Title: Fool for the City, Categories: CDs & Vinyl, Classic Rock, Album-Oriented Rock (AOR), Score: 2.013
ID: 8617, Title: Gulag Orkestar, Categories: CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi, Indie Rock, Score: 1.998
ID: 3167, Title: Operation Mindcrime Lp (Original Issue), Categories: CDs & Vinyl, Rock, Progressive, Progressive Rock, Score: 1.956
ID: 2885, Title: Daytime Friends-The Best of Kenny Rogers, Categories: CDs & Vinyl, Country, Today's Country, Score: 1.876
ID: 6923, Title: Alternative Christmas Album / Various, Categories: CDs & Vinyl, Pop, Vocal Pop, Score: 1.793


In [67]:
recs = get_top_k(model, dataset, train, user_id=18628, k=5)
for item in recs:
    print(f"ID: {item['iid']}, Title: {item['title']}, Categories: {item['categories']}, Score_FM: {item['score']:.3f}")
    

ID: 18766, Title: Joe Pass: Portraits of Duke Ellington [ LP Vinyl ], Categories: CDs & Vinyl, Vinyl Store, Score_FM: 1.992
ID: 2150, Title: Fool for the City, Categories: CDs & Vinyl, Classic Rock, Album-Oriented Rock (AOR), Score_FM: 1.981
ID: 8617, Title: Gulag Orkestar, Categories: CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi, Indie Rock, Score_FM: 1.935
ID: 2317, Title: Source Presents: Hip Hop Hits, Vol. 7       Clean Version, Categories: CDs & Vinyl, R&B, Soul, Score_FM: 1.848
ID: 2685, Title: Mizerable 2, Categories: CDs & Vinyl, International Music, Far East & Asia, Japan, Score_FM: 1.830


From this line

In [ ]:
# user_recs = {}
# for uid in train['uid'].unique():
#     recs = get_top_k(model, dataset, train, user_id=uid, k=50)
#     user_recs[uid]=recs

In [74]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(Settings(
    persist_directory="chroma_storage",  
    anonymized_telemetry=False
))

collection = chroma_client.get_or_create_collection("recommendations")

In [76]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid

model_embedding = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def cast_metadata(metadata):
    return {
        k: (
            int(v) if isinstance(v, (np.integer,))
            else float(v) if isinstance(v, (np.floating,))
            else str(v) if isinstance(v, (list, dict))
            else v
        )
        for k, v in metadata.items()
    }

def prepare_data_for_insertion(user_recs):
    combined_texts, metadata_list, id_list = [], [], []

    for user_id, recs in user_recs.items():
        for rec in recs:
            text = f"Title: {rec['title']}. Categories: {rec['categories']}"
            metadata = cast_metadata({
                "user_id": str(user_id),
                "iid": rec['iid'],
                "title": rec['title'],
                "categories": rec['categories'],
                "cf_score": rec['score'],
                "combined_text": text
            })

            combined_texts.append(text)
            metadata_list.append(metadata)
            id_list.append(str(uuid.uuid4()))

    return combined_texts, metadata_list, id_list

def batch_insert_with_parallelism(model, collection, user_recs, batch_size=512, max_workers=4):
    texts, metadatas, ids = prepare_data_for_insertion(user_recs)

    # Batch encode using SentenceTransformer
    print("Encoding embeddings...")
    embeddings = model.encode(texts, batch_size=128, show_progress_bar=True)

    # Split into batches
    total = len(texts)
    batches = range(0, total, batch_size)

    print("Inserting into ChromaDB...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(batches, desc="Inserting batches"):
            futures.append(executor.submit(
                collection.add,
                documents=texts[i:i+batch_size],
                embeddings=embeddings[i:i+batch_size],
                metadatas=metadatas[i:i+batch_size],
                ids=ids[i:i+batch_size]
            ))

        for f in futures:
            f.result()  

In [77]:
def query_recommendations_by_text(user_query: str, user_id: str, model, collection, top_k=10):
    query_embedding = model.encode(user_query).tolist()

    # query ChromaDB with user filter
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"user_id": {"$eq": str(user_id)}},
        include=["metadatas", "documents", "distances"]
    )

    retrieved = []
    if results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["ids"][0])):
            meta = results["metadatas"][0][i]
            retrieved.append({
                "iid": meta["iid"],
                "title": meta["title"],
                "categories": meta["categories"],
                "cf_score": meta["cf_score"],
                "similarity_score": 1 - results["distances"][0][i]
            })

    return retrieved

In [80]:
print("Running insert...")
batch_insert_with_parallelism(model_embedding, collection, user_recs)

Running insert...
Encoding embeddings...


Batches:   0%|          | 0/12388 [00:00<?, ?it/s]

Inserting into ChromaDB...


Inserting batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 3097/3097 [00:01<00:00, 2252.31it/s]


In [81]:
collection.count()

1585614

In [103]:
user_query = "recommend some love songs"
beauty_users = sampled_train[sampled_train['categories'].str.contains("beauty", case=False, na=False)]['uid'].unique()

beauty_user_ids = [uid for uid in beauty_users if uid in user_recs]
user_id = beauty_user_ids[0] if beauty_user_ids else next(iter(user_recs.keys()))


results = query_recommendations_by_text(user_query, str(user_id), model_embedding, collection, top_k=10)

for r in results:
    print(f"Title: {r['title']}, Categories: {r['categories']}, Similarity: {r['similarity_score']:.3f}, CF Score: {r['cf_score']:.3f}")

In [85]:
peeked = collection.peek()

print("Number of documents in peek:", len(peeked["ids"]))

if "metadatas" in peeked and len(peeked["metadatas"]) > 0:
    print("Example user_id in ChromaDB metadata:")
    print(peeked["metadatas"][0]["user_id"]) 
else:
    print("No metadata found in peek.")


Number of documents in peek: 10
Example user_id in ChromaDB metadata:
592


In [86]:
print("Chosen user_id:", user_id)
print("Type:", type(user_id))

Chosen user_id:
32
Type: <class 'numpy.int64'>


In [89]:
user_id = peeked["metadatas"][0]["user_id"]
results = query_recommendations_by_text(user_query, str(user_id), model_embedding, collection, top_k=10)

In [91]:
user_query = " "
user_id = peeked["metadatas"][0]["user_id"]
results = query_recommendations_by_text(user_query, user_id, model_embedding, collection, top_k=10)
print("user_id:", user_id)
print("Type of user_id:", type(user_id))

if not results:
    print("No recommendations found.")
else:
    print(f"Top {len(results)} results for user_id: {user_id}")
    for r in results:
        print(f"Title: {r['title']}, Categories: {r['categories']}, Similarity: {r['similarity_score']:.3f}, CF Score: {r['cf_score']:.3f}")

user_id: 592
Type of user_id: <class 'str'>
No recommendations found.


In [93]:
matches = collection.get(limit=1, include=["metadatas", "documents"])
sample_meta = matches["metadatas"][0]

print("Sample metadata:", sample_meta)
print("Stored user_id:", sample_meta.get("user_id"), "Type:", type(sample_meta.get("user_id")))

Sample metadata: {'user_id': '592', 'iid': 191722, 'title': 'Dancing Fantasy', 'categories': 'CDs & Vinyl, Country, Bluegrass', 'cf_score': 0.7575958371162415, 'combined_text': 'Title: Dancing Fantasy. Categories: CDs & Vinyl, Country, Bluegrass'}
Stored user_id: 592 Type: <class 'str'>


In [97]:
matches = collection.get(where={"user_id": {"$eq": "592"}}, include=["metadatas"])
print("Number of records for user_id='592':", len(matches["ids"]))

Number of records for user_id='592': 15


In [99]:
results = collection.query(
    query_embeddings=[model_embedding.encode(user_query).tolist()],
    n_results=10,
    include=["metadatas", "documents", "distances"]
)
print(results)

{'ids': [['e6bcc6a1-a1f4-43a2-9042-77680d7598dd', 'f6528e0d-3a11-4631-9018-063b1124aa69', '97b5b024-00e7-4330-b11e-0640439a1cbb', '6098c7d4-9078-4f1e-8573-3fcb2f8d677c', '8ec5a681-583f-4394-9a84-8ea46dff1e29', '3b49a8e6-6ca3-4074-a1f9-7642fac71d12', '0d7522d9-d4b5-4d48-af6b-f8e44f400577', '8939ed17-42c8-463a-a121-783e2e8f4b86', '621c0122-4118-4d62-9678-b7aa4add59a9', '0c7af11e-70f7-4134-b2ad-ea29a69bde7a']], 'embeddings': None, 'documents': [['Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Jazz', 'Title: Love Will Find a Way. Categories: CDs & Vinyl, Ja

In [104]:
user_query = "recommend some different love songs"

# Run the raw query without filtering by user_id
query_embedding = model_embedding.encode(user_query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=["metadatas", "documents", "distances"]
)

print(f"\n User query: \"{user_query}\"\n")

for i, meta in enumerate(results["metadatas"][0]):
    cosine_similarity = 1 - results["distances"][0][i]
    print(f"Result #{i+1}")
    print(f"  User ID: {meta['user_id']}")
    print(f"  Item ID: {meta['iid']}")
    print(f"  Title: {meta['title']}")
    print(f"  Categories: {meta['categories']}")
    print(f"  CF Score: {meta['cf_score']:.3f}")
    print(f"  Cosine Similarity: {cosine_similarity:.3f}")
    print("-" * 50)


 User query: "recommend some different love songs"

Result #1
  User ID: 46214
  Item ID: 1542
  Title: Get Closer
  Categories: CDs & Vinyl, Country, Today's Country
  CF Score: 0.442
  Cosine Similarity: -0.155
--------------------------------------------------
Result #2
  User ID: 46701
  Item ID: 1542
  Title: Get Closer
  Categories: CDs & Vinyl, Country, Today's Country
  CF Score: 0.812
  Cosine Similarity: -0.155
--------------------------------------------------
Result #3
  User ID: 46720
  Item ID: 1542
  Title: Get Closer
  Categories: CDs & Vinyl, Country, Today's Country
  CF Score: 0.931
  Cosine Similarity: -0.155
--------------------------------------------------
Result #4
  User ID: 47080
  Item ID: 1542
  Title: Get Closer
  Categories: CDs & Vinyl, Country, Today's Country
  CF Score: 0.961
  Cosine Similarity: -0.155
--------------------------------------------------
Result #5
  User ID: 47100
  Item ID: 1542
  Title: Get Closer
  Categories: CDs & Vinyl, Country, 

In [73]:
import os
import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter

model_embeddings = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [75]:
def batch_query_user_recommendations(user_recs, model, collection, top_k=10):
    queries = []
    user_ids = []

    for user_id, recs in user_recs.items():
        if recs:
            top_rec = recs[0]
            query_text = f"Title: {top_rec['title']}. Categories: {top_rec['categories']}"
            queries.append(query_text)
            user_ids.append(str(user_id))

    query_embeddings = model.encode(queries, batch_size=128, show_progress_bar=True)

    result_dict = {}

    for i in tqdm(range(len(user_ids)), desc="Querying users"):
        results = collection.query(
            query_embeddings=[query_embeddings[i]],
            n_results=top_k,
            where={"user_id": {"$eq": user_ids[i]}},
            include=["metadatas", "documents", "distances"]
        )

        if results and results["metadatas"][0]:
            result_dict[user_ids[i]] = [
                {
                    "iid": meta["iid"],
                    "title": meta["title"],
                    "categories": meta["categories"],
                    "cf_score": meta["cf_score"],
                    "similarity_score": 1 - dist
                }
                for meta, dist in zip(results["metadatas"][0], results["distances"][0])
            ]

    return result_dict


In [ ]:
def prep_docs_pinecone(user_id, recs):
    docs = []
    ids = []
    metadatas = []

    for cand in recs:
        combined_text = f"Title: {cand['title']}. Categories: {cand['categories']}"
        chunks = text_splitter.split_text(combined_text)

        for i, chunk in enumerate(chunks):
            chunk_id = f"{item['iid']}_{i}"

            meta_data = {
                "iid": cand['iid'],
                "title": cand['title'],
                "categories": cand['categories'],
                "combined_text": cand['combined_text"],
                "fm_score": cand['score'],
                "chunk": cand['chunk']
            }

            ids.append(chunk_id)
            docs.append(chunks)
            metadatas.append(meta_data)
            
    return ids, docs, metadatas        

In [81]:
os.environ["PINECONE_API_KEY"]= "pcsk_6mDSEp_B8EBirbyCBt5wUsw8QDFrwp8Z1jeyGvjytLpabnV2my2tJSZ1eNgCYt5xmcwgzk"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# project_info = pc.describe_project()
# print("Allowed cloud+regions:", project_info.cloud_region_defaults)

# Create index if not exists
index_name = "recommendations-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine"     
    )

index = pc.Index(index_name)

# pinecone.init(api_key="pcsk_6mDSEp_B8EBirbyCBt5wUsw8QDFrwp8Z1jeyGvjytLpabnV2my2tJSZ1eNgCYt5xmcwgzk", environment="YOUR_ENVIRONMENT")
# index_name = "recommendations-index"

# if index_name not in pinecone.list_indexes():
#     pinecone.create_index(name=index_name,
#                           dimension=768,
#                           metric="cosine")

# index = pinecone.Index(index_name)

PINECONE_API_KEY_ID = "pcsk_6mDSEp_B8EBirbyCBt5wUsw8QDFrwp8Z1jeyGvjytLpabnV2my2tJSZ1eNgCYt5xmcwgzk"

TypeError: Pinecone.create_index() missing 1 required positional argument: 'spec'

In [ ]:
ids, docs, metadatas = prep_docs_pinecone(train['uid'], recs)

In [82]:
collection.count()

0

In [ ]:
batch_size = 32
for i in range(0, len(docs), batch_size):
    batch_ids = ids[i:i+batch_size]
    batch_docs = docs[i:i+batch_size]
    batch_metas = metadatas[i:i+batch_size]

    embeddings = model.encode(batch_docs)
    vectors = []

    for id, embedding, meta in zip(batch_ids, embeddings, batch_meatas):
        vectors.append((id, embedding.tolist(), meta))

    index.upsert(vectors = vectors)

In [ ]:
def get_search_result(user_query, index, model, k):
    query_embedding = model.encode(user_query).to_list()

    result = index.query(vector=query_embedding,
                        topk=k,
                        include_metadata=True,
                        include_values=False)
    search_results = []
    for match in search_results:
        metadata = match['metadata']
        search_results.append({
            "iid": metadata['iid'],
            "title": metadata['title'],
            "categories": metadata['categories'],
            "combined_text": metadata['combined_text'],
            "fm_score": metadata['fm_score'],
            "similarity_score": metadata['score']
        })

    return search_results
                                   

In [79]:
user_query = "beauty magazine that can provide beauty tips"
beauty_users = sampled_train[sampled_train['categories'].str.contains("beauty", case=False, na=False)]['uid'].unique()

beauty_user_ids = [uid for uid in beauty_users if uid in user_recs]
user_id = beauty_user_ids[0] if beauty_user_ids else next(iter(user_recs.keys()))


results = query_recommendations_by_text(user_query, user_id, model_embedding, collection, top_k=10)

for r in results:
    print(f"Title: {r['title']}, Categories: {r['categories']}, Similarity: {r['similarity_score']:.3f}, CF Score: {r['cf_score']:.3f}")

In [81]:
for r in results:
    print(f"Title: {r['title']}, Categories: {r['categories']}, Similarity: {r['similarity_score']:.3f}, CF Score: {r['cf_score']:.3f}")

In [80]:
print(results)

[]


In [ ]:
from sentence_transformers import SentenceTransformer
import uuid

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def cast_metadata(metadata):
    return {
        k: (
            int(v) if isinstance(v, (np.integer,))
            else float(v) if isinstance(v, (np.floating,))
            else str(v) if isinstance(v, (list, dict))
            else v
        )
        for k, v in metadata.items()
    }

for user_id, recs in user_recs.items():
    for rec in recs:
        combined_text = f"Title: {rec['title']}. Categories: {rec['categories']}"
        embedding = model.encode(combined_text).tolist()

        metadata = cast_metadata({
            "user_id": str(user_id),
            "iid": rec['iid'],
            "title": rec['title'],
            "categories": rec['categories'],
            "cf_score": rec['score'],
            "combined_text": combined_text
        })

        collection.add(
            documents=[combined_text],
            embeddings=[embedding],
            ids=[str(uuid.uuid4())],
            metadatas=[metadata]
        )

In [ ]:
def query_user_recommendations(user_id, top_k=10):
    # Use one of the top LightFM items for this user to build the query
    # You can also use a user profile embedding if available
    top_rec = user_recs[user_id][0]  # or some average over top recs
    query_text = f"Title: {top_rec['title']}. Categories: {top_rec['categories']}"
    query_embedding = model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"user_id": {"$eq": str(user_id)}},  # Optional: filter only this user's candidates
        include=["metadatas", "documents", "distances"]
    )

    # Combine results into a list of dicts
    retrieved = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        retrieved.append({
            "iid": meta["iid"],
            "title": meta["title"],
            "categories": meta["categories"],
            "cf_score": meta["cf_score"],
            "similarity_score": 1 - results["distances"][0][i],  # Convert cosine distance to similarity
        })

    return retrieved


In [ ]:
print(train[['uid', 'iid', 'title']].head())
print(f"Unique users: {train['uid'].nunique()}")
print(f"Unique items: {train['iid'].nunique()}")

In [ ]:
small_train = train.head(1000)  # Test with smaller dataset
model, dataset, _ = fit_and_train(small_train)
recs = get_top_k(model, dataset, small_train, user_id=1, k=3)

In [ ]:
# user_id_encoder = dict(zip(merged_df['user_id_encoded'], merged_df['user_id']))
recs_dict = {user_id: get_top_k(model, dataset, train, user_id, k=50) for user_id in train['uid'].unique()}

In [ ]:
def get_purchase_history(merged_df, user_id, k):
    user_hist = merged_df[merged_df['user_id_encoded']==user_id]
    user_hist_sorted = user_hist.sort_values(by='timestamp', ascending=False)
    recent_purchase = user_hist_sorted[['asin', 'title_meta', 'description', 'combined_text']].head(k)
    return recent_purchase

def get_like_items(merged_df, user_id):
    like_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==True)].sort_values(by='timestamp', ascending=False)
    like_items = like_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

def get_dislike_items(merged_df, user_id):
    dislike_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==False)].sort_values(by='timestamp', ascending=False)
    dislike_items = dislike_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

In [ ]:
def evaluate_recall(model, train_interactions, test_interactions, k):
    precision= precision_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    recall= recall_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    auc= auc_score(model, test_interactions, train_interactions=train_interactions).mean()
    return precision, recall, auc

eval_metrics = evaluate_recall(model, train_interactions, test_interactions, k=50)

In [ ]:
def get_top_k(model, dataset, merged_df, user_id, k):
    n_users, n_items = dataset.interactions_shape()

    user_ids = np.full(n_items, user_id)  # or use np.repeat(user_id, n_items)
    # item_ids = np.arange(n_items)
    interacted_items = set(merged_df[merged_df['user_id_encoded']==user_id]['asin_encoded'])

    scores = model.predict(user_ids, np.arange(n_items))
    scores[list(interacted_items)] = -np.inf

    top_items_encoded = np.argsort(-scores)[:k]
    asin_info = dict(zip(merged_df['asin_encoded'], zip(merged_df['asin'], merged_df['title_meta'], merged_df['combined_text'])))
    recs = [asin_info[item] for item in top_items_encoded if item in asin_info]
    return recs

# user_id_encoder = dict(zip(merged_df['user_id_encoded'], merged_df['user_id']))
recs_dict = {user_id: get_top_k(model, dataset, merged_df, user_id, k=50) for user_id in merged_df['user_id_encoded'].unique()}

def get_recs(user_id):
    return recs_dict.get(user_id, [])

In [ ]:
user_id_test = merged_df['user_id_encoded'].iloc[8]
recent_purchases = get_purchase_history(merged_df, user_id_test, k=10)
like_items = get_like_items(merged_df, user_id_test)
dislike_items = get_dislike_items(merged_df, user_id_test)
top_k_recs= get_recs(user_id_test)

In [ ]:
# save
save_path = "../../data/amazon"

save_in_chunks(train, os.path.join(save_path, "train_amazon.pkl"), chunk_size=1000)
save_in_chunks(val, os.path.join(save_path, "val_amazon.pkl"), chunk_size=1000)
save_in_chunks(test, os.path.join(save_path, "test_amazon.pkl"), chunk_size=1000)

Saved 0 rows. Current memory usage: 970.27 MB
Saved 10000 rows. Current memory usage: 833.91 MB
Saved 20000 rows. Current memory usage: 827.78 MB
Saved 30000 rows. Current memory usage: 792.19 MB
Saved 40000 rows. Current memory usage: 799.44 MB
Saved 50000 rows. Current memory usage: 808.91 MB
Saved 60000 rows. Current memory usage: 831.64 MB
Saved 70000 rows. Current memory usage: 921.38 MB
Saved 80000 rows. Current memory usage: 957.84 MB
Saved 90000 rows. Current memory usage: 993.12 MB
Saved 100000 rows. Current memory usage: 1023.00 MB
Saved 110000 rows. Current memory usage: 1055.94 MB
Saved 120000 rows. Current memory usage: 1088.94 MB
Saved 130000 rows. Current memory usage: 1046.52 MB
Saved 140000 rows. Current memory usage: 1058.25 MB
Saved 150000 rows. Current memory usage: 1058.50 MB
Saved 160000 rows. Current memory usage: 1084.78 MB
Saved 170000 rows. Current memory usage: 1073.72 MB
Saved 180000 rows. Current memory usage: 1103.55 MB
Saved 190000 rows. Current memory us

# Remove from here

In [6]:
# save
save_path = "../../data/amazon/saved/"

train.to_pickle(save_path + "train_amazon.pkl")
val.to_pickle(save_path + "valid_amazon.pkl")
test.to_pickle(save_path + "test_amazon.pkl")

NameError: name 'train' is not defined

In [ ]:
# merge df vs meta
merged_df = user_df.merge(meta_df, on='parent_asin', how='right', suffixes=('', '_meta'))
merged_df = merged_df.rename(columns={'title': 'title_review'})
merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='s')
merged_df

,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_meta,average_rating,rating_number,features,description,categories,details
0,1.0,Weak,Definitely the weakest of the three albums SWV...,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,2015-01-03 06:22:17.000,1.0,0.0,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
1,1.0,Too Many Guest Spots,Unlike It's About Time and New Beginning this ...,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,2013-09-16 20:19:59.000,1.0,1.0,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
2,4.0,"When &quot;Tension&quot; is released, expect q...","Their third and final album as a group, &quot;...",B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,2000-03-19 00:44:40.000,1.0,0.0,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
3,5.0,Thanks,thank you,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,2015-03-27 18:15:21.000,0.0,1.0,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
4,2.0,Release Some Tension - 2.5 stars,When I was trying to decide whether or not to ...,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,2010-11-13 08:44:58.000,3.0,0.0,Release Some Tension,4.6,112,,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827554,5.0,A Singer of Distinction,"Dame Joan Hilda Hood Hammond, DBE, CMG (24 May...",B000T001IM,B000T001IM,AFICGLJ62Y3YRTYN7PNYM6UUOA6Q,2013-12-20 11:54:02.000,1.0,1.0,"Joan Hammond, Historical Recordings from 1941-49",5.0,1,,,"CDs & Vinyl, Classical",{}
4827555,4.0,Great Performer!,I first saw Heinz Winckler in a Broadway music...,B00069I6RO,B00069I6RO,AHLRFMDIVAQF7ARTPM6PVJO6DTYQ,2008-01-03 08:10:12.000,2.0,1.0,Come Alive,4.5,4,,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout...","{""Package Dimensions"": ""5.55 x 4.97 x 0.54 inc..."
4827556,5.0,Cheaper,"OK, loves, it's cheaper to go to amazon.co.uk ...",B00069I6RO,B00069I6RO,AGZVQUGBE64F3M2A5EBU2CNDTKIA,2004-12-15 22:34:52.000,5.0,0.0,Come Alive,4.5,4,,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout...","{""Package Dimensions"": ""5.55 x 4.97 x 0.54 inc..."
4827557,5.0,Beautiful and poignant.,Wonderful collection of songs with haunting tu...,B08BF2PH1X,B08BF2PH1X,AEZET6CFIEJWTU7UTEK57IDARX2A,2021-02-09 22:59:55.740,0.0,1.0,Long Day in the Milky Way,4.4,16,,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk","{""Product Dimensions"": ""5.5 x 4.9 x 0.4 inches..."


In [ ]:
# filter merge_df with rating >= 3
merged_df['like'] = merged_df['rating']>3

# data.drop_duplicates(subset=['asin'])

In [ ]:
# encode user_id and asin
merged_df['user_id_encoded'] = LabelEncoder().fit_transform(merged_df['user_id'])
merged_df['asin_encoded'] = LabelEncoder().fit_transform(merged_df['asin'])
merged_df['parent_asin_encoded'] = LabelEncoder().fit_transform(merged_df['parent_asin'])

# data partition
# merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)
df_train, df_test = train_test_split(merged_df, test_size=0.2, random_state=42)
train_interactions_set = set(zip(df_train['user_id_encoded'], df_train['asin_encoded']))
df_test = df_test[~df_test[['user_id_encoded', 'asin_encoded']].apply(tuple, axis=1).isin(train_interactions_set)]

## Lightfm

In [ ]:
dataset = Dataset()
dataset.fit(users= merged_df['user_id_encoded'], items = merged_df['asin_encoded'])
train_interactions, train_w = dataset.build_interactions((row['user_id_encoded'], row['asin_encoded']) for id, row in df_train.iterrows())
test_interactions, test_w = dataset.build_interactions((row['user_id_encoded'], row['asin_encoded']) for id, row in df_test.iterrows())

In [ ]:
model = LightFM(loss='warp', item_alpha=1e-6, user_alpha=1e-6)
# train model
model.fit(train_interactions, epochs=50, num_threads=2, sample_weight=train_w)

In [ ]:
def add_combined_text(row):
    combined_text = f"title_review: {str(row['title_review'])}, text: {str(row['text'])}, title_meta: {str(row['title_meta'])}, features: {str(row['features'])}, description: {str(row['description'])}, details: {str(row['details'])}, categories: {str(row['categories'])}"
    row['combined_text'] = combined_text
    return row

# data = merged_df.map(add_combined_text)
merged_df = merged_df.apply(add_combined_text, axis=1)
# combined_texts = merged_df["combined_text"]
merged_df

,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_meta,...,rating_number,features,description,categories,details,like,user_id_encoded,asin_encoded,parent_asin_encoded,combined_text
0,3.0,Three Stars,all advertizements,B00FA7T630,B00FA7T630,AFNRZBYAPLFKJNJU5BK3TTRWHJ6A,2015-10-26 17:49:26.000,0,1,GQ Print Access Print Magazine,...,10,,"Product Description, Dive into, GQ, ’s culture...",,"{""Date First Available"": ""June 2, 2020"", ""Manu...",False,24328,2499,2499,"title_review: Three Stars, text: all advertize..."
1,5.0,A great audiophile audio magazine that covers ...,A great audiophile audio magazine that covers ...,B00F8P62PO,B00F8P62PO,AETSCPCEEZ3S3BGNU4WCIN4H62ZQ,2018-01-11 07:17:33.014,6,1,Hi-Fi + Print Magazine,...,44,,Hi-Fi+ is Europe's premier English-language hi...,"Magazine Subscriptions, Arts, Music & Photogra...","{""Date First Available"": ""September 19, 2013"",...",True,12133,2498,2498,title_review: A great audiophile audio magazin...
2,3.0,inspirational,i really like the ideas they have in this maga...,B003F1W9T6,B003F1W9T6,AGQCHHMSO6S2DJLWGGFLCVUY477Q,2011-12-24 03:36:48.000,2,1,Paper Crafts,...,3,,,,"{""Date First Available"": ""May 12, 2021""}",False,40505,2095,2095,"title_review: inspirational, text: i really li..."
3,2.0,Should be called card making,"I ordered this magazine because I enjoy ""paper...",B003F1W9T6,B003F1W9T6,AEIKS6KU32T7VBIAP7YIMR2LICQQ,2013-03-03 05:28:59.000,0,1,Paper Crafts,...,3,,,,"{""Date First Available"": ""May 12, 2021""}",False,6799,2095,2095,"title_review: Should be called card making, te..."
4,4.0,Four Stars,Son loves this,B00007AXX1,B00007AXX1,AGDDHCNIGZPKBQRST6UVTCVPD53A,2015-12-04 12:24:30.000,0,1,Horse Illustrated,...,284,,,"Magazine Subscriptions, Sports, Recreation & O...","{""Package Dimensions"": ""10.79 x 8.11 x 0.31 in...",True,34411,871,871,"title_review: Four Stars, text: Son loves this..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71492,1.0,"Beautiful fashion magazine, subscription never...",I have been subscribing to V for a few years n...,B0007INIGI,B0007INIGI,AFDU3G7WMDWWA2XJLYNQLHSIEZMQ,2012-03-01 16:47:32.000,1,1,V Magazine - Ny Print Magazine,...,5,,"V Magazine, was launched in September 1999 as ...","Magazine Subscriptions, Fashion & Style, Men","{""Date First Available"": ""September 14, 2006"",...",False,19700,1315,1315,"title_review: Beautiful fashion magazine, subs..."
71493,1.0,What a fiasco,I subscribed to this magazine a year ago and n...,B0002800W8,B0002800W8,AG5BWPCEIS6FEJKQZTLJVQLGQNGA,2007-09-05 15:08:34.000,2,0,Victorian Review Print Magazine,...,1,,"Published since 1972,, Victorian Review, is an...",,"{""Date First Available"": ""September 14, 2006"",...",False,31564,1248,1248,"title_review: What a fiasco, text: I subscribe..."
71494,1.0,Have yet to receive the first issue,It has been a month since I purchased this mag...,B00007L0RC,B00007L0RC,AHA6K63RE5ODXSTDNPRND73U3EMQ,2011-02-15 16:55:54.000,0,1,Visto Print Magazine,...,2,,"A news magazine targeted at the wider public, ...",,"{""Date First Available"": ""September 15, 2013"",...",False,47920,1032,1032,title_review: Have yet to receive the first is...
71495,4.0,Super Fast Delivery,I was totally surprised how fast I received th...,B00007L0RC,B00007L0RC,AGQT3FF5NHZUOS3YSEAMIRY65VRQ,2012-01-18 22:58:43.000,0,1,Visto Print Magazine,...,2,,"A news magazine targeted at the wider public, ...",,"{""Date First Available"": ""September 15, 2013"",...",True,40744,1032,1032,"title_review: Super Fast Delivery, text: I was..."


In [ ]:
def get_purchase_history(merged_df, user_id, k):
    user_hist = merged_df[merged_df['user_id_encoded']==user_id]
    user_hist_sorted = user_hist.sort_values(by='timestamp', ascending=False)
    recent_purchase = user_hist_sorted[['asin', 'title_meta', 'description', 'combined_text']].head(k)
    return recent_purchase

def get_like_items(merged_df, user_id):
    like_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==True)].sort_values(by='timestamp', ascending=False)
    like_items = like_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

def get_dislike_items(merged_df, user_id):
    dislike_items = merged_df[(merged_df['user_id_encoded']==user_id) & (merged_df['like']==False)].sort_values(by='timestamp', ascending=False)
    dislike_items = dislike_items[['asin', 'title_meta', 'description']].iloc[:10]
    return like_items

In [ ]:
def evaluate_recall(model, train_interactions, test_interactions, k):
    precision= precision_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    recall= recall_at_k(model, test_interactions, train_interactions=train_interactions, k=k).mean()
    auc= auc_score(model, test_interactions, train_interactions=train_interactions).mean()
    return precision, recall, auc

eval_metrics = evaluate_recall(model, train_interactions, test_interactions, k=50)

In [ ]:
print(eval_metrics)

(0.010412165, 0.20115773521154753, 0.85380054)


In [ ]:
def get_top_k(model, dataset, merged_df, user_id, k):
    n_users, n_items = dataset.interactions_shape()

    user_ids = np.full(n_items, user_id)  # or use np.repeat(user_id, n_items)
    # item_ids = np.arange(n_items)
    interacted_items = set(merged_df[merged_df['user_id_encoded']==user_id]['asin_encoded'])

    scores = model.predict(user_ids, np.arange(n_items))
    scores[list(interacted_items)] = -np.inf

    top_items_encoded = np.argsort(-scores)[:k]
    asin_info = dict(zip(merged_df['asin_encoded'], zip(merged_df['asin'], merged_df['title_meta'], merged_df['combined_text'])))
    recs = [asin_info[item] for item in top_items_encoded if item in asin_info]
    return recs

# user_id_encoder = dict(zip(merged_df['user_id_encoded'], merged_df['user_id']))
recs_dict = {user_id: get_top_k(model, dataset, merged_df, user_id, k=50) for user_id in merged_df['user_id_encoded'].unique()}

def get_recs(user_id):
    return recs_dict.get(user_id, [])

In [ ]:
user_id_test = merged_df['user_id_encoded'].iloc[8]
recent_purchases = get_purchase_history(merged_df, user_id_test, k=10)
like_items = get_like_items(merged_df, user_id_test)
dislike_items = get_dislike_items(merged_df, user_id_test)
top_k_recs= get_recs(user_id_test)

In [ ]:
top_k_recs

[('B00007B1JS',
  'Soap Opera Digest',
  'title_review: Subscription, text: I am unable to review this magazine  I have never received an issue.<br /><br />Needless to say, I am not happy about this.<br /><br />Lynda Epler, title_meta: Soap Opera Digest, features: , description: , details: {"Date First Available": "April 30, 2015"}, categories: Magazine Subscriptions, Entertainment & Pop Culture, Television'),
 ('B00GBJIEQG',
  'Inked',
  'title_review: Five Stars, text: Great magazine, title_meta: Inked, features: , description: , details: {"Date First Available": "October 31, 2013"}, categories: Magazine Subscriptions, Professional & Educational Journals, Professional & Trade, Arts, Photography'),
 ('B00007EN1O',
  'Alpaca    Print Magazine',
  'title_review: Excellent publication about alpacas, text: This magazine is a must for any alpaca owner, would be alpaca owner, or one who is just learning about these beautiful animals.<br />It is beautifully done, showing pictures of alpacas,

In [ ]:
user_id_test = merged_df['user_id_encoded'].iloc[8]
recent_purchases = get_purchase_history(merged_df, user_id_test, k=3)
recent_purchases

,asin,title_meta,description,combined_text
8,B00007AXX1,Horse Illustrated,,"title_review: Great Magazine, text: This magaz..."


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 6.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.document_loaders import TextLoader
from langchain.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:

# since max number of tokens of sentence transformers is 512
# split the text into smaller chunks that fit within this limit before embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# prepare documents
def prepare_documents(user_id_test, recs):
    documents = []
    for asin, title_meta, combined_text in recs:
        chunks = text_splitter.split_text(combined_text)
        for chunk in chunks:
            metadata = {
                "asin": asin,
                "title_meta": title_meta,
            }
            doc = Document(page_content=chunk, metadata=metadata)
            documents.append(doc)
    return documents

user_documents = prepare_documents(user_id_test, top_k_recs)
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN_HERE"
# YOUR_HUGGINGFACE_TOKEN_HERE
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
chroma_client = Chroma.from_documents(documents=user_documents, embedding=embeddings, persist_directory="./content/chroma_db")

def get_search_result(user_query, chroma_client, k):
    res = chroma_client.similarity_search(query=user_query, k=k)
    return res



<ipython-input-25-0d67646e6ac0>:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
user_query= "beauty magazine that can provide beauty tips"
retrieved_docs = get_search_result(user_query, chroma_client, k=5)
retrieved_docs

[Document(metadata={'asin': 'B00005QJE2', 'title_meta': 'Nylon    Print Magazine'}, page_content='interested in a fashion magazine that goes beyond a "beach read.", title_meta: Nylon    Print Magazine, features: , description: A vibrant and proactive voice for today\'s hip, intelligent, young women seeking fresh perspectives on fashion, beauty and music., details: {"Date First Available": "September 14, 2006", "Manufacturer": "Nylon Media Inc."}, categories: Magazine Subscriptions, Business & Investing, International'),
 Document(metadata={'asin': 'B00005QJE2', 'title_meta': 'Nylon    Print Magazine'}, page_content='title_review: inspiring, text: as a high school student aspiring to attend art school, nylon has provided me with countless sources of inspiration in their creative photography, layout, and decoration. i look forward to reading the magazine each month, and i honestly devour every last word and picture.<br /><br />nylon is a creative, whimsical and fantastic magazine. i high

In [ ]:
# !pip install accelerate

In [ ]:
model_id = "nvidia/Mistral-NeMo-Minitron-8B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# dtype = torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(model_id, load_in_8bit=True)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
)

# HuggingFace pipeline integration
hf = HuggingFacePipeline(pipeline=pipe)
llm = hf

tokenizer_config.json:   0%|          | 0.00/177k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
prompt_template = """
Imagine you are a recommender. Given the user's recent purchase history, liked items, disliked items, and query:
User Query: {question}
Recent Purchases: {purchase_history}
Liked Items: {liked_items}
Disliked Items: {disliked_items}

Please rank the following items based on relevance and provide a brief explanation for each:
{context}
Reranked Recommendations:
"""

def personal_rerank(user_query, recent_purchase, retrieved_docs, like_items, dislike_items, llm, prompt_template):
    purchase_hist = "\n".join([f"{i+1}. {row['title_meta']}: {row['description']}" for i, (_, row) in enumerate(recent_purchases.iterrows())])
    context = "\n".join(f"{doc.metadata['title_meta']}: {doc.page_content}" for doc in retrieved_docs)
    prompt = prompt_template.format(
        question=user_query,
        purchase_history=purchase_hist,
        liked_items="\n".join(like_items),
        disliked_items="\n".join(dislike_items),
        context=context
    )
    return llm(prompt)


In [ ]:
user_query = "beauty magazine that can provide beauty tips"
retrieved_docs = get_search_result(user_query, chroma_client, k=5)
